Full example of DEM process, with a single example pointing. We are using the second pointing on June 3rd, 2022. This pointing was observed for two orbits, and has two different ARs which can be studied. It also suffers ghost ray contamination, allowing for demonstration of the correction method. 

In [ ]:
import numpy as np
from astropy import units as u
import pickle
import importlib

In [ ]:
#System-specific paths; edit for you + your machine.

#email for jsoc notification:
jsoc_email='jessie.m.duncan@nasa.gov'
#Path to top-level do-dem directory - edit for your system.
path_to_dodem = '/Users/jmdunca2/do-dem/'
#Path to folder containing NuSTAR data products – should be downloaded, decompressed + have already had the initial pipeline run.
nustar_folder = '/Users/jmdunca2/nustar/jun-2022/'


from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

import all_nu_analysis as ana

In [ ]:
parent_dir = path_to_dodem+'/FPExample/'
targets_file = parent_dir+'/example_targets.pickle'
dem_folder = parent_dir+'/DEM_folders/'

#reference file path:
reference_path = parent_dir+'/reference_files/'


all_targets = {}
firsttime=True #Set False if you want to edit the manual entry fields of the pointing directory without erasing all entries added later.

In [ ]:
key = '03-jun-22_2'

pointing_dict = {'NOAA_ARID': ['AR13026', 'AR13027'], #Identified NOAA ARIDs (from e.g. helioviewer)
                  'HARP': [8281, 8278], #Identified SHARP numbers (from e.g. helioviewer)
                  'datapaths': [nustar_folder+'/20624004001/',
                                 nustar_folder+'/20624006001/'],
                  'obsids': ['20624004001', '20624006001'],
                  'working_dir': dem_folder+'/initial_dem_3jun22_2/',
                  'loc': ['limb', 'disk'], #From examining the location of the ARs, e.g. from https://github.com/ianan/nsigh_all/blob/master/maps/maps_20220603/maps_20220603_213250_nu20624006001_FPMA.png
                  'method': 'input', #We will manually input chosen regions for this pointing, as it's not a good candidate for gaussian fitting to the regions.
                  'notes': '',
                  'goes_satellite': 16} #What GOES sattelite was active at this time?

if firsttime:
    all_targets[key] = pointing_dict
else:
    all_targets[key].update(pointing_dict)



In [ ]:
#Get statistics about the exposure time.
ana.get_exposures(all_targets, dogoes=False)

In [ ]:
#Saving dictionary with all information to the file assigned at the beginning
with open(targets_file, 'wb') as f:
         # Pickle the 'data' dictionary using the highest protocol available.
         pickle.dump(all_targets, f, pickle.HIGHEST_PROTOCOL)   

In [ ]:
#Manual entry regions: a list of dictionaries containing the center and radius of circular regions.
#Note that the file starter_region.reg is required in this process – this can be any region file (it just needs to be in the correct region 
#file format). 

regions = [{'centerx': 900.0, 'centery': 230.0, 'radius': 150.0},
              {'centerx': 420.0, 'centery': -300.0, 'radius': 150.0}]

ana.manual_prep(key, targets_file,
                    plot=True, 
                    make_scripts=True,
                    inputregion=regions, write_input_regions=True,
                    path_to_dodem=path_to_dodem,
                    targets_file=targets_file,
                   parent_dir=parent_dir)

Above, we see the selected regions (and the results of two-gaussian region fitting) for each orbit. We see that the fitting process finds the two regions we are interested in correctly in only one of the orbits (this is the reason we are using the manual input approach). 

This function has also written a bash script, entitled "run_ar_tis_03-jun-22_2.sh" located in the scripts directory. The purpose of the script is to run TIME INTERVAL SELECTION for each region for this pointing. The next step is to (outside of this notebook) go and run that script. It may be necessary to change the permissions of the script in order to run it. It is important to do this in a bash session where heasoft has been initialized. On this machine, that requires the commands:

bash

heainit


Note that the file fvth_out.dat is also required in this process, if it is not present it will be made via a call to f_vth.pro (but this requires a working IDL license, as the script will do this in IDL. 


The time interval selection process sometimes needs to be run more than once to properly finish (it's suspected that this has something to do with errors from running multiple instances of the NuSTAR pipeline elements concurrently – but this has not been confirmed. You can confirm that the process is complete when there is a new pickle file for each orbit for each region, which contains lists of the selected time intervals. 

Note that data products for unsuccessful time intervals may be made along the way, in addition to the sucessful ones. In order to clean up the results, the following is run (the unsuccesful interval directories can safely be deleted, as they will not be used). 

- The info-dump section saves some new information about the chosen time intervals to the targets dictionary. This include determining which of the new intervals are flare-time intervals, and which are quiet time intervals. It does this by referencing several different saved lists of flare times in the reference files dictionary (flare-time finding and/or quiescent-time verification is not demonstrated here). 

- The clean up function moves non-used time interval directories out of the way.

In [ ]:
ana.post_tis_info_dump(key, targets_file, flarepath=reference_path)
ana.clean_up_post_tis(key, targets_file)

NuSTAR data products are now ready for the DEM process. We need to do a few more steps to prepare to use AIA data, and to add a ghost ray correction if necessary. 

In [ ]:
#Let's take a look at the total key list now:
with open(targets_file, 'rb') as f:
    all_targets = pickle.load(f)


ARDict = all_targets[key]
print(ARDict.keys())

#Let's fetch some AIA files (one per orbit) to use in co-aligning NuSTAR to AIA:
aiamaps = ana.sample_aia(ARDict, jsoc_email=jsoc_email)

Using the "shift_" defined below (in x,y arcseconds), we'll make a 6-panel plot that shows:

- unshifted NuSTAR data with data region
- shifted NuSTAR data (to match AIA, using the input shift)
- unshifted NuSTAR data with both regions (AIA and NuSTAR extraction regions, which will be shifted with respect to each other also by shift_)

This can be repeated with changes to the shift_ values until you are satisfied with the result. 

Note that only one of the two regions will be plotted at a time, but the shift needs to be the same for both. The first region will be plotted in the manner described for all orbits, followed by the second (or further). It's important to verify that the chosen shift provides reasonable co-alignment for all regions. 

In [ ]:
shift_ = [-50, 50]

#Do region adjustment
import images_and_coalignment as iac


aagt = ARDict['per_region_all_time_intervals'] 
method = ARDict['method']

if method=='double':
    gauss_stats = ARDict['gauss_stats']
    sep_axis = gauss_stats[0][0]
else:
    sep_axis = ''

regdicts_all = []
for j in range(0, len(aagt)):
    regdicts=[]
    for i in range(0, len(ARDict['datapaths'])):
        #print('AAGT:', aagt[j][i])
        if not aagt[j][i]:
            regdicts.append([])
            continue
        aati = iac.per_orbit_region_adjustment(ARDict, i, aiamaps, 
                                                    nushift=shift_, method=method, shush=True, 
                                                    sep_axis=sep_axis, show=True,
                                                    pick_region=True, regionind=j)
        print(j)
        #append new region dictionary to list
        if method == 'fit':
                regdicts.append(aati['aiaregdicts'][0])
        else:
            regdicts.append(aati['aiaregdicts'][0])

    regdicts_all.append(regdicts)



#Updates to obs-key dictionary:
ARDict['nushift'] = shift_
ARDict['regdicts'] = regdicts_all
all_targets[key] = ARDict

with open(targets_file, 'wb') as f:
    # Pickle the 'data' dictionary using the highest protocol available.
    pickle.dump(all_targets, f, pickle.HIGHEST_PROTOCOL) 

Before we do the DEMs themselves, it's also important to add a ghost ray correction (given that contamination is visible for this pointing. This will take a few minutes the first time it's run, as extracting the ghost ray counts by energy involves making full-orbit NuSTAR spectral data products.

In [ ]:
import ghost_correct as gc

#Choose DEM energy ranges for NuSTAR
eng_tr = [[2.5, 3.5], [3.5, 6.], [6., 10.]]

#Choose ghost-ray extraction region – this is the worst identifiable contamination that is well-separated from a real source. 
gregion = {'centerx': 250.0, 'centery': -75.0, 'radius': 150.0}
#Choose the orbit during which the ghost ray contamination is the worst. 
gorbit=0
#Get the region finalized before switching to True (as the process after that point is 
go_ahead=True
write_new_file=True


gc.make_ghost_region(key, targets_file, gregion=gregion, gorbit=gorbit, 
                     go_ahead=go_ahead, 
                     write_new_file=write_new_file)


Now, to actually do the DEMs. For this example, we will be retrieving AIA data from the JSOC. To save time and to save on the considerable AIA data volume, we will only do DEMs for the quiet time intervals. 

In [ ]:
#Re-load the targets file, to make sure it's updated with everything needed (including ghost ray correction info)
with open(targets_file, 'rb') as f:
    data = pickle.load(f)

ARDict = data[key]
#print(ARDict.keys())


prqti = ARDict['per_region_quiet_time_intervals']
#print([prqti])

importlib.reload(ana)

for r in range(0, len(prqti)):
    ana.do_key_dem(key, targets_file,
                   ghost_corr = True, #Ghost ray correction
                   extraname='ghost_corr', #Add string to result files to indicate there was a ghost ray correction
                   asym_stdv_uncert=True, #Newer DEM uncertainty method
              high_temp_analysis=False, #Set true to compute DEMs with three extra temperature bounds
              aia_region_dict=ARDict['regdicts'], #List of AIA regions - input for data prep
              use_prepped_aia=False,
              input_time_intervals=prqti, #Time intervals to do DEMs for
              pick_region=True, #whether or not to do this only for one selected region (i.e. of several in the FOV)
              regionind=r, #Index of desired region
              do_no_xrt_version=False, #Next three keys: options for DEMs with only some instruments.
              do_onlyaia_version=False,
              do_aiaxrt_version=False)